In [0]:
%run ./_utils_config

In [0]:
%run ./_utils_bronze_framework

In [0]:
%run ./_utils_onboarding_tools

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Schema ready: {catalog}.{schema}")

In [0]:
print("Available source specs:", list_source_specs())

ree_spec = load_source_spec_file("ree")

errors = validate_source_spec(ree_spec)
print("✅ Spec valid" if not errors else f"❌ {errors}")

In [0]:
import copy


def broken_spec_errors(change, base_spec=None):
    spec = copy.deepcopy(base_spec or ree_spec)
    change(spec)
    return validate_source_spec(spec)


negative_tests = {
    "http instead of https": lambda s: s["endpoints"][0].update(url="http://apidatos.ree.es/es/datos"),
    "internal host": lambda s: s["endpoints"][0].update(url="https://localhost/admin"),
    "unknown placeholder": lambda s: s["endpoints"][0]["params"].update(token="{api_key}"),
    "POST method": lambda s: s["endpoints"][0].update(method="POST"),
    "query string in url": lambda s: s["endpoints"][0].update(url="https://apidatos.ree.es/x?a=1"),
    "unsupported source_type": lambda s: s.update(source_type="jdbc"),
    "missing source_type": lambda s: s.pop("source_type"),
    "csv_delimiter on json endpoint": lambda s: s["endpoints"][0].update(csv_delimiter=";"),
    "initial_window too large": lambda s: s["load"].update(initial_window=5000),
    "refresh_window > initial_window": lambda s: s["load"].update(refresh_window=200),
    "unit not allowed": lambda s: s["load"].update(unit="week"),
    "chunk_size > 1 without {end}": lambda s: s["endpoints"][0]["params"].pop("end_date"),
    "full strategy with unit": lambda s: s["load"].update(strategy="full"),
    "old 'incremental' block": lambda s: (s.pop("load"), s.update(incremental={"type": "date_range"})),
    "end_offset out of range": lambda s: s["load"].update(end_offset=30),
}

for test_name, change in negative_tests.items():
    errors = broken_spec_errors(change)
    print(f"{'✅ detected' if errors else '❌ NOT DETECTED'} · {test_name}: {errors}")

In [0]:
summary = run_source_spec(ree_spec, window_override=14)
print(f"\n{summary['chunks_ok']} ok / {summary['chunks_error']} errors · run_id {summary['run_id']}")
display(dbutils.fs.ls(f"{bronze_volume_path}/ree/prices"))

In [0]:
display(
    spark.table(ingestion_log_table)
    .orderBy(F.desc("ingested_at"), "endpoint_name", "chunk_start")
    .limit(100)
)

In [0]:
# Bronze tables test: run twice and check rows are replaced, not duplicated
def bronze_counts():
    counts = {}
    for ep in ree_spec["endpoints"]:
        table = bronze_table_name(ree_spec["source_name"], ep["name"])
        df = spark.table(table)
        counts[table] = (df.count(), df.select("source_file").distinct().count())
    return counts

run_source_spec(ree_spec, window_override=14)
counts_1 = bronze_counts()
run_source_spec(ree_spec, window_override=14)
counts_2 = bronze_counts()

print()
for table, (rows, files) in counts_2.items():
    print(f"{table}: {rows} rows / {files} files")
print(f"{'✅' if counts_1 == counts_2 else '❌'} Same row counts after re-running (no duplicates)")
print(f"{'✅' if all(r == f for r, f in counts_2.values()) else '❌'} One row per JSON file")

In [0]:
# Calendar periods: chunk boundaries for day / month / year (pure logic, no downloads)
def check(label, actual, expected):
    print(f"{'✅' if actual == expected else '❌'} {label}: {actual}" + ("" if actual == expected else f"  (expected {expected})"))

endpoint = {"name": "test"}

# Day: 7-day chunks are Monday-Sunday weeks
day_spec = {"load": {"strategy": "incremental", "unit": "day", "initial_window": 14, "refresh_window": 7,
                     "chunk_size": 7, "end_offset": 0}}
check("day chunks (run on Mon 2026-11-16)",
      [(str(s), str(e)) for s, e in build_chunks(day_spec, endpoint, date(2026, 11, 16))],
      [("2026-11-02", "2026-11-08"), ("2026-11-09", "2026-11-15"), ("2026-11-16", "2026-11-16")])

# Month: 2-month chunks aligned to the calendar, current month up to today
month_spec = {"load": {"strategy": "incremental", "unit": "month", "initial_window": 3, "refresh_window": 1,
                       "chunk_size": 2, "end_offset": 0}}
chunks_a = build_chunks(month_spec, endpoint, date(2026, 11, 16))
chunks_b = build_chunks(month_spec, endpoint, date(2026, 11, 20))
check("month chunks (run on 2026-11-16)",
      [(str(s), str(e)) for s, e in chunks_a],
      [("2026-07-01", "2026-08-31"), ("2026-09-01", "2026-10-31"), ("2026-11-01", "2026-11-16")])
check("month chunks keep the same starts on another day", [s for s, _ in chunks_a], [s for s, _ in chunks_b])

# Year: holidays spec, 3 years back + current + next
holidays_spec = load_source_spec_file("public_holidays_es")
check("year chunks (holidays)",
      [str(s) for s, _ in build_chunks(holidays_spec, holidays_spec["endpoints"][0], date(2026, 11, 16), 3)],
      ["2023-01-01", "2024-01-01", "2025-01-01", "2026-01-01", "2027-01-01"])

# Gap healing: missed runs widen the refresh window, never above initial_window
check("REE after a 20-day gap", resolve_window(ree_spec, date.today() + timedelta(days=20)), (21, "refresh"))
check("holidays in mid-2028 (one year missed)", resolve_window(holidays_spec, date(2028, 6, 1)), (2, "refresh"))

In [0]:
reset_probe_budget()

result = probe_url(
    "https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real",
    {"start_date": "2026-11-10T00:00", "end_date": "2026-11-10T23:59", "time_trunc": "hour"},
)
print(json.dumps({k: v for k, v in result.items() if k != "sample"}, indent=2, ensure_ascii=False))

In [0]:
result = probe_url("https://date.nager.at/api/v3/PublicHolidays/2026/ES")
print(json.dumps({k: v for k, v in result.items() if k != "sample"}, indent=2, ensure_ascii=False))

In [0]:
safety_tests = {
    "http instead of https": ("http://date.nager.at/api/v3/PublicHolidays/2026/ES", None),
    "internal host": ("https://localhost/admin", None),
    "query string in url": ("https://date.nager.at/api/v3/PublicHolidays/2026/ES?x=1", None),
    "boolean param": ("https://date.nager.at/api/v3/PublicHolidays/2026/ES", {"flag": True}),
}

for test_name, (url, params) in safety_tests.items():
    result = probe_url(url, params)
    print(f"{'✅ rejected' if result['status'] == 'rejected' else '❌ NOT REJECTED'} · {test_name}: {result.get('errors')}")

In [0]:
result = probe_url("https://date.nager.at/api/v3/ThisDoesNotExist")
print(result["status"], result["http_status"], "· probes left:", result["probes_left"])

In [0]:
# propose_source_spec checks: none of these must reach the human approval step
# a) Broken YAML: unquoted {start}
bad_yaml = """
source_name: test_bad_yaml
source_type: http_api
description: test
load:
  strategy: incremental
  unit: day
  initial_window: 7
  refresh_window: 1
  chunk_size: 7
endpoints:
  - name: prices
    url: https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real
    response_format: json
    params:
      start_date: {start}T00:00
      end_date: {end}T23:59
"""
print("a)", json.loads(propose_source_spec(bad_yaml))["stage"], "\n")

# b) Valid YAML, but the source already exists
print("b)", json.loads(propose_source_spec(spec_to_yaml(load_source_spec_file("ree"))))["errors"], "\n")

# c) Valid spec, but the API path does not exist -> fails in the dry run
not_found_yaml = """
source_name: test_not_found
source_type: http_api
description: "Test: endpoint that does not exist"
load:
  strategy: full
endpoints:
  - name: nothing
    url: https://date.nager.at/api/v3/ThisDoesNotExist
    method: GET
    response_format: json
"""
print("c)", json.loads(propose_source_spec(not_found_yaml))["stage"])

In [0]:
import copy

locations_test_yaml = """
source_name: test_locations
source_type: http_api
description: "Test: Open-Meteo archive for the 52 province capitals"
load:
  strategy: incremental
  unit: day
  initial_window: 7
  refresh_window: 1
  chunk_size: 7
  end_offset: -5
  date_format: "%Y-%m-%d"
endpoints:
  - name: hourly_weather
    url: https://archive-api.open-meteo.com/v1/archive
    method: GET
    response_format: json
    params:
      start_date: "{start}"
      end_date: "{end}"
      hourly: temperature_2m,wind_speed_10m,shortwave_radiation
      timezone: GMT
    locations:
      reference: locations_es.csv
      key_column: province_code
      latitude_param: latitude
      longitude_param: longitude
"""
locations_spec = load_source_spec(locations_test_yaml)

errors = validate_source_spec(locations_spec)
print("✅ Spec with locations valid" if not errors else f"❌ {errors}")

location_negative_tests = {
    "reference file does not exist": lambda s: s["endpoints"][0]["locations"].update(reference="missing.csv"),
    "path in reference": lambda s: s["endpoints"][0]["locations"].update(reference="../secrets.csv"),
    "unknown key_column": lambda s: s["endpoints"][0]["locations"].update(key_column="city_code"),
    "latitude also in params": lambda s: s["endpoints"][0]["params"].update(latitude="40.4"),
    "missing longitude_param": lambda s: s["endpoints"][0]["locations"].pop("longitude_param"),
}

for test_name, change in location_negative_tests.items():
    spec = copy.deepcopy(locations_spec)
    change(spec)
    errors = validate_source_spec(spec)
    print(f"{'✅ detected' if errors else '❌ NOT DETECTED'} · {test_name}: {errors}")

In [0]:
location_params, location_codes = resolve_locations(locations_spec["endpoints"][0])
print(f"{len(location_codes)} locations · first: {location_codes[:3]} · last: {location_codes[-3:]}")
print("latitude param (start):", location_params["latitude"][:60], "...")
print()

summary = run_source_spec(locations_spec, window_override=6, dry_run=True)
for r in summary["results"]:
    ok = r["top_level_items"] == len(location_codes)
    print(f"{'✅' if ok else '❌'} {r['chunk_start']}->{r['chunk_end']}: "
          f"{r['top_level_items']} location blocks returned (expected {len(location_codes)})")
print()
print(summary["results"][0]["sample"][:300])